# Balance Sheet Analysis & Fair Value Estimation

- **Input**: `Reports/balance_sheet.csv` (from `company_report_autofetch.py`, amounts in millions)
- **Output**: `Reports/complete_company_analysis.xlsx`
  - `1_Historical_All_Quarters`: per-quarter ratios, growth and TTM metrics
  - `2_Latest_Quarter_Complete`: latest quarter per stock + price-based valuation (used by `app.py` and scoring)

Input contract: the notebook computes most columns itself, but `OperatingMargin` and `BVPS` (book value per share)
must already exist as raw columns in `balance_sheet.csv` — if the autofetch ever drops them this notebook fails fast
with a `KeyError` instead of silently producing wrong ratios.

### Setup
Load `Reports/balance_sheet.csv` (quarterly statements from `company_report_autofetch.py`) and the latest prices.

In [ ]:
import os

import numpy as np
import pandas as pd
import yfinance as yf

import sector_mapping

REPORTS_DIR = sector_mapping.REPORTS_DIR  # absolute, next to sector_mapping.py
df = pd.read_csv(os.path.join(REPORTS_DIR, "balance_sheet.csv"), parse_dates=["FiscalDateEnding"])
df = df.drop(columns=["Date", "DateAdded"]).sort_values(["Symbol", "FiscalDateEnding"]).reset_index(drop=True)
# Total debt / cash are fetched by company_report_autofetch.py from now on; older cached rows do not have them yet
for col in ["TotalDebt", "CashAndEquivalents"]:
    if col not in df.columns:
        df[col] = np.nan
print(f"Rows with TotalDebt: {df['TotalDebt'].notna().sum()} / {len(df)}")
df = df.drop_duplicates(["Symbol", "FiscalDateEnding"], keep="last").reset_index(drop=True)
# A missing balance-sheet value in one quarter is carried forward from the previous quarter (max 1 quarter)
BALANCE_ITEMS = ["TotalAssets", "TotalLiabilities", "TotalShareholderEquity", "CommonStockSharesOutstanding",
                 "TotalDebt", "CashAndEquivalents"]
df[BALANCE_ITEMS] = df.groupby("Symbol")[BALANCE_ITEMS].ffill(limit=1)
print(f"Loaded {len(df)} rows, {df['Symbol'].nunique()} stocks")

## 1. Quarterly Ratios, Growth & TTM Metrics

TTM income items = sum of the last 4 quarters; TTM balance items = average of the first and last quarter in that window.

In [ ]:
shares = df["CommonStockSharesOutstanding"].replace(0, np.nan)
pos_equity = df["TotalShareholderEquity"].where(df["TotalShareholderEquity"] > 0)  # ROE etc. meaningless on negative equity

# Profitability
df["ROE"] = (df["NetIncome"] / pos_equity * 100).round(2)
df["ROA"] = (df["NetIncome"] / df["TotalAssets"] * 100).round(2)
df["NetProfitMargin"] = (df["NetIncome"] / df["TotalRevenue"] * 100).round(2)
df["GrossMargin"] = (df["GrossProfit"] / df["TotalRevenue"] * 100).round(2)

# Per share
df["EPS"] = (df["NetIncome"] / shares).round(2)
df["RevenuePerShare"] = (df["TotalRevenue"] / shares).round(2)
df["AssetsPerShare"] = (df["TotalAssets"] / shares).round(2)
df["OperatingIncomePerShare"] = (df["OperatingIncome"] / shares).round(2)

# Efficiency & health
df["AssetTurnover"] = (df["TotalRevenue"] / df["TotalAssets"]).round(2)
df["EquityTurnover"] = (df["TotalRevenue"] / pos_equity).round(2)
# Debt metrics: interest-bearing TotalDebt when available, else total liabilities (flagged in Debt_Metric_Basis)
equity = df["TotalShareholderEquity"].replace(0, np.nan)
has_debt = df["TotalDebt"].notna()
debt_or_liab = df["TotalDebt"].where(has_debt, df["TotalLiabilities"])
df["Debt_Metric_Basis"] = np.where(has_debt, "TotalDebt", "TotalLiabilities")
df["Debt_to_Equity"] = (debt_or_liab / equity).round(2)
df["Liabilities_to_Equity"] = (df["TotalLiabilities"] / equity).round(2)
df["DebtToAssets"] = (debt_or_liab / df["TotalAssets"] * 100).round(2)
df["LiabilitiesToAssets"] = (df["TotalLiabilities"] / df["TotalAssets"] * 100).round(2)
df["EquityRatio"] = (df["TotalShareholderEquity"] / df["TotalAssets"] * 100).round(2)
df["EquityMultiplier"] = (df["TotalAssets"] / pos_equity).round(2)
df["ROIC_Approx"] = (df["OperatingIncome"] / df["TotalAssets"] * 100).round(2)
df["OperatingToNetIncome"] = (df["OperatingIncome"] / df["NetIncome"]).round(2)
df["DebtToEBITDA_Approx"] = (debt_or_liab / df["OperatingIncome"].abs()).round(2)

# Growth (QoQ = previous quarter, YoY = same quarter last year). Rows are only compared when the fiscal dates are
# really ~1 quarter / ~1 year apart (a missing quarter no longer shifts the comparison), and the base is |previous|
# so a move from a loss to a profit shows as positive growth (pct_change flipped the sign).
g = df.groupby("Symbol")


def growth(col, periods, days):
    prev = g[col].shift(periods)
    gap = (df["FiscalDateEnding"] - g["FiscalDateEnding"].shift(periods)).dt.days
    return ((df[col] - prev) / prev.abs().replace(0, np.nan) * 100).where(gap.between(days * 0.75, days * 1.25))


for name, col in [("Revenue", "TotalRevenue"), ("NetIncome", "NetIncome"), ("EPS", "EPS")]:
    df[f"{name}Growth_QoQ"] = growth(col, 1, 91).round(2)
    df[f"{name}Growth_YoY"] = growth(col, 4, 365).round(2)
df["OperatingLeverage"] = (growth("OperatingIncome", 1, 91) / growth("TotalRevenue", 1, 91).replace(0, np.nan)).round(2)
# Clean inf BEFORE the rolling windows below: an inf would poison rolling means/sums
df = df.replace([np.inf, -np.inf], np.nan)

# 4-quarter windows
g = df.groupby("Symbol")  # re-group: df was replaced (inf -> NaN) above, so the earlier grouper is stale
roll4 = lambda col, fn: g[col].transform(lambda s: getattr(s.rolling(4, min_periods=1), fn)())
df["AvgNetMargin_4Q"] = roll4("NetProfitMargin", "mean").round(2)
df["AvgOperatingMargin_4Q"] = roll4("OperatingMargin", "mean").round(2)
# TTM = sum of the last 4 quarters when they are consecutive (span <= ~300 days). With 2-3 quarters (new listings)
# the available quarters are annualized (x 4/n) and TTM_Quarters shows how many were used; otherwise NaN.
# (Old code summed 1-3 quarters as if they were a full year and filled missing TTM with 0.)
span_ok = (df["FiscalDateEnding"] - g["FiscalDateEnding"].shift(3)).dt.days.le(300) | g.cumcount().lt(3)
df["TTM_Quarters"] = g["TotalRevenue"].transform(lambda s: s.rolling(4, min_periods=1).count())
for ttm, col in [("TTM_Revenue", "TotalRevenue"), ("TTM_NetIncome", "NetIncome"),
                 ("TTM_OperatingIncome", "OperatingIncome"), ("TTM_GrossProfit", "GrossProfit")]:
    n = g[col].transform(lambda s: s.rolling(4, min_periods=1).count())
    df[ttm] = (roll4(col, "sum") * 4 / n.replace(0, np.nan)).where((n >= 2) & span_ok)

pos = g.cumcount()
def window_avg(col):
    first = g[col].shift(3).where(pos >= 3, g[col].transform(lambda s: s.iloc[0]))
    return (first + df[col]) / 2

avg_equity, avg_assets = window_avg("TotalShareholderEquity"), window_avg("TotalAssets")
df["TTM_ROE"] = (df["TTM_NetIncome"] / avg_equity.where(avg_equity > 0) * 100).round(2)
df["TTM_ROA"] = (df["TTM_NetIncome"] / avg_assets * 100).round(2)
df["TTM_NetProfitMargin"] = (df["TTM_NetIncome"] / df["TTM_Revenue"] * 100).round(2)
df["TTM_OperatingMargin"] = (df["TTM_OperatingIncome"] / df["TTM_Revenue"] * 100).round(2)
df["TTM_GrossMargin"] = (df["TTM_GrossProfit"] / df["TTM_Revenue"] * 100).round(2)
df["TTM_EPS"] = (df["TTM_NetIncome"] / shares).round(2)
df["TTM_RevenuePerShare"] = (df["TTM_Revenue"] / shares).round(2)
df["TTM_AssetTurnover"] = (df["TTM_Revenue"] / avg_assets).round(2)
df["TTM_EquityTurnover"] = (df["TTM_Revenue"] / avg_equity.where(avg_equity > 0)).round(2)
df = df.replace([np.inf, -np.inf], np.nan)
print(f"✅ {len(df.columns)} historical columns")

## 2. Fair Value Estimation (latest quarter)

1. **Sector-peer P/E, P/B, P/S** (global median when a sector has < 3 peers)
2. **Growth-adjusted P/E** (fair P/E = 2 × EPS growth, clipped to 8–35)
3. **Graham** intrinsic value and growth value
4. **Owner-earnings DCF** (2-stage, 9% discount, 2.5% terminal)
5. **Composite**: weighted mean of methods within 0.25×–4× price, weights renormalized over valid methods
6. **Range**: FairValue_Low / Composite / FairValue_High, MarginOfSafety, Valuation_Confidence (method agreement)

Price = latest Close in `signal_analysis.csv`; symbols missing there fall back to a 5-day yfinance download
(a symbol can be missing if it was added to the universe after the last signal run — without a price,
no price-based valuation is possible, so those rows are valued only once a price exists).

In [ ]:
latest = df.groupby("Symbol").tail(1).reset_index(drop=True)

# --- Price ---
sig = pd.read_csv(os.path.join(REPORTS_DIR, "signal_analysis.csv"), usecols=["Symbol", "Date", "Close"])
price = sig.sort_values("Date").groupby("Symbol")["Close"].last()
# Fallback price for symbols missing from signal_analysis.csv.
# (A single-ticker download comes back as a Series, hence the to_frame.)
missing = sorted(set(latest["Symbol"]) - set(price.index))
yf_close = pd.Series(dtype=float)
if missing:
    try:
        px_ = yf.download(missing, period="5d", progress=False, auto_adjust=True)["Close"]
        if isinstance(px_, pd.Series):
            px_ = px_.to_frame(missing[0])
        yf_close = px_.ffill().iloc[-1]
        print(f"yfinance filled {yf_close.notna().sum()}/{len(missing)} missing prices")
    except Exception as e:  # network / yfinance changes: value only the symbols we have prices for
        print(f"⚠️ yfinance download failed ({e}); {len(missing)} symbols left without price")
latest["CurrentPrice"] = latest["Symbol"].map(pd.concat([price, yf_close]))

v = latest[latest["CurrentPrice"].notna()].copy()
v["Sector"] = v["Symbol"].map(sector_mapping.symbol_sector).fillna("Unknown")
px = v["CurrentPrice"]

# --- Ratios ---
v["PE_Ratio"] = (px / v["TTM_EPS"]).where(v["TTM_EPS"] > 0).round(2)
v["PB_Ratio"] = (px / v["BVPS"]).replace([np.inf, -np.inf], np.nan).round(2)
v["PS_Ratio"] = (px / v["TTM_RevenuePerShare"]).replace([np.inf, -np.inf], np.nan).round(2)
v["PEG_Ratio"] = (v["PE_Ratio"] / v["EPSGrowth_YoY"]).where(v["EPSGrowth_YoY"] > 0).round(2)

def sector_median(col):
    """Sector median of col; global median when the sector has < 3 values."""
    by_sector = v[col].groupby(v["Sector"])
    return by_sector.transform("median").where(by_sector.transform("count") >= 3, v[col].median())

v["Sector_Median_PE"] = sector_median("PE_Ratio").where(v["PE_Ratio"].notna())
v["Sector_Median_PB"] = sector_median("PB_Ratio")
v["Sector_Median_PS"] = sector_median("PS_Ratio").where(v["PS_Ratio"].notna())

# --- Fair value legs ---
eps, bvps, eps_g = v["TTM_EPS"], v["BVPS"], v["EPSGrowth_YoY"]
v["FairValue_PE"] = (eps * v["Sector_Median_PE"]).where(eps > 0).round(2)
fair_pe_growth = (eps_g.clip(0, 20) * 2).clip(8, 35).round(2).where(eps_g > 0)
v["FairValue_GrowthPE"] = (eps * fair_pe_growth).where(eps > 0).round(2)
v["FairValue_PB"] = (bvps * v["Sector_Median_PB"]).round(2)
v["FairValue_PS"] = (v["TTM_RevenuePerShare"] * v["Sector_Median_PS"]).round(2)
v["Graham_IntrinsicValue"] = np.sqrt((22.5 * eps * bvps).clip(lower=0)).where((eps > 0) & (bvps > 0)).round(2)
v["Graham_GrowthValue"] = (eps * (8.5 + 2 * eps_g.clip(0, 15))).where((eps > 0) & eps_g.notna()).round(2)

# Owner-earnings DCF: TTM net income if > 0, else 70% of TTM operating income
owner_earnings = v["TTM_NetIncome"].where(v["TTM_NetIncome"] > 0, (v["TTM_OperatingIncome"] * 0.70).where(v["TTM_OperatingIncome"] > 0))
oe_ps = owner_earnings / v["CommonStockSharesOutstanding"].replace(0, np.nan)
dcf_growth = (v["RevenueGrowth_YoY"].fillna(eps_g).clip(-5, 15) / 100).fillna(0.03)  # named to not shadow the growth() helper
R_DISC, G_TERM, YEARS = 0.09, 0.025, 5

def dcf_price(oe0, rate):
    if pd.isna(oe0) or oe0 <= 0:
        return np.nan
    fcf, pv = float(oe0), 0.0
    for t in range(1, YEARS + 1):
        fcf *= 1 + rate
        pv += fcf / (1 + R_DISC) ** t
    return round(pv + fcf * (1 + G_TERM) / (R_DISC - G_TERM) / (1 + R_DISC) ** YEARS, 2)

v["FairValue_DCF"] = [dcf_price(o, gr) for o, gr in zip(oe_ps, dcf_growth)]

# --- Size ---
v["MarketCap"] = (px * v["CommonStockSharesOutstanding"]).round(2)
# EV = market cap + total debt - cash (only when debt is available); the old proxy is kept under an honest name
v["EnterpriseValue"] = (v["MarketCap"] + v["TotalDebt"] - v["CashAndEquivalents"].fillna(0)).where(v["TotalDebt"].notna()).round(2)
v["MarketCap_Plus_Liabilities"] = (v["MarketCap"] + v["TotalLiabilities"]).round(2)
v["EV_Revenue_Ratio"] = (v["EnterpriseValue"] / v["TTM_Revenue"]).replace([np.inf, -np.inf], np.nan).round(2)
v["EarningsYield"] = (eps / px * 100).where(eps > 0).round(2)
v["BookToMarket"] = (bvps / px).where(bvps > 0).round(2)
v["POI_Ratio"] = (px / (v["TTM_OperatingIncome"] / v["CommonStockSharesOutstanding"])).where(v["TTM_OperatingIncome"] > 0).round(2)

#### Composite fair value
Blend the individual fair-value estimates into one number per stock (weights differ for profitable and loss-making companies).

Only methods within 0.25×–4× of the current price count (outliers are discarded, weights renormalized over the survivors);
if none qualify, all positive methods are used at capped confidence. **Unprofitable floor:** for loss-making companies the
composite is floored at 50% of the P/B fair value — a company with real book value shouldn't be valued near zero just
because the earnings-based methods collapse.

In [ ]:
# --- Composite ---
PROFIT_WEIGHTS = {"FairValue_PE": 0.25, "FairValue_GrowthPE": 0.15, "FairValue_PB": 0.15,
                  "Graham_IntrinsicValue": 0.15, "FairValue_DCF": 0.30}
UNPROF_WEIGHTS = {"FairValue_PB": 0.35, "FairValue_PS": 0.25, "FairValue_DCF": 0.40}
is_profitable = eps > 0


def composite_row(row, weights):
    """Weighted mid of methods within 0.25x-4x price; else all positive methods at low confidence."""
    current = row["CurrentPrice"]  # local name: must not shadow the price Series from the price cell above
    picked = {c: row[c] for c in weights if pd.notna(row[c]) and 0.25 * current <= row[c] <= 4.0 * current}
    fallback = not picked
    if fallback:
        picked = {c: row[c] for c in weights if pd.notna(row[c]) and row[c] > 0}
    if not picked:
        return np.nan, np.nan, np.nan, np.nan, 0.0
    num = den = 0.0
    for c, val in picked.items():
        num += val * weights[c]
        den += weights[c]
    vals = np.array(list(picked.values()), dtype=float)
    mid = num / den
    disp = float(vals.std() / vals.mean()) if vals.mean() > 0 and len(vals) > 1 else 0.0
    conf = 1.0 / (1.0 + disp)
    if fallback:
        conf = min(conf, 0.35)
    return round(mid, 2), round(vals.min(), 2), round(vals.max(), 2), round(disp, 3), round(max(conf, 0.25), 3)


cols = ["FairValue_Composite", "FairValue_Low", "FairValue_High", "MethodDispersion", "Valuation_Confidence"]
v[cols] = [composite_row(r, PROFIT_WEIGHTS if p else UNPROF_WEIGHTS) for (_, r), p in zip(v.iterrows(), is_profitable)]

# Unprofitable floor: at least 50% of PB fair value
unprof = ~is_profitable
no_mid = v["FairValue_Composite"].isna()
v.loc[unprof, "FairValue_Composite"] = np.maximum(v.loc[unprof, "FairValue_Composite"].fillna(0), v.loc[unprof, "FairValue_PB"].fillna(0) * 0.5)
v.loc[unprof & (v["FairValue_Composite"] <= 0), "FairValue_Composite"] = np.nan
floor_only = unprof & no_mid & v["FairValue_Composite"].notna()
floor_val = v.loc[floor_only, "FairValue_Composite"]
v.loc[floor_only, "FairValue_Low"] = floor_val
v.loc[floor_only, "FairValue_High"] = floor_val
v.loc[floor_only, ["MethodDispersion", "Valuation_Confidence"]] = [0.0, 0.25]
# The floor can lift the composite above the original method max (e.g. ZENA: Low=1.94, Composite=5.99, High=1.94):
# keep the published range bracketing the composite (display only; the composite itself is unchanged).
bracket = unprof & v["FairValue_Composite"].notna()
v.loc[bracket, "FairValue_High"] = v.loc[bracket, ["FairValue_High", "FairValue_Composite"]].max(axis=1)
v.loc[bracket, "FairValue_Low"] = v.loc[bracket, ["FairValue_Low", "FairValue_Composite"]].min(axis=1)

fv = v["FairValue_Composite"].where(v["FairValue_Composite"] > 0)
v["Price_vs_FairValue"] = ((px / fv - 1) * 100).round(2)
v["MarginOfSafety"] = ((fv - px) / fv * 100).round(2)
v["Valuation_Signal"] = np.select(
    [v["Price_vs_FairValue"] < -10, v["Price_vs_FairValue"] > 10, v["Price_vs_FairValue"].notna()],
    ["Undervalued", "Overvalued", "Fairly Valued"], "Cannot Determine",
)
v["Profitability_Status"] = np.where(is_profitable, "Profitable",
                                     np.where(eps.isna(), "No Data", "Unprofitable"))  # NaN EPS != unprofitable; floor/math unchanged

print(f"Stocks valued: {len(v)} | Profitable: {is_profitable.sum()} | Unprofitable: {unprof.sum()}")
print(v["Valuation_Signal"].value_counts().to_string())
v[["Symbol", "Sector", "CurrentPrice", "FairValue_Low", "FairValue_Composite", "FairValue_High",
   "MarginOfSafety", "Valuation_Signal", "Valuation_Confidence"]].sort_values("MarginOfSafety", ascending=False).head(20)

## 3. Export

In [ ]:
valuation_cols = [c for c in v.columns if c not in latest.columns]
latest_complete = latest.merge(v[["Symbol", *valuation_cols]], on="Symbol", how="left")

excel_file = os.path.join(REPORTS_DIR, "complete_company_analysis.xlsx")
with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="1_Historical_All_Quarters", index=False)
    latest_complete.to_excel(writer, sheet_name="2_Latest_Quarter_Complete", index=False)
print(f"✅ Sheet 1: {len(df)} rows × {len(df.columns)} columns")
print(f"✅ Sheet 2: {len(latest_complete)} stocks × {len(latest_complete.columns)} columns")